Goal: Traffic Flow Optimization in Austin

1. Variables : Green light time for each direction g(i)
2. Constraints: Cycle time(L) = G ns + G ew + clear / limit light time / traffic inflow Capacity.
3. Objective : Minimize Delay Time

In [ ]:
#!pip install pyomo
#import sys
#!{sys.executable} -m pip install pyomo
#!pip install cyipopt

In [1]:
#Import library
import pyomo.environ as aml

In [2]:
#Generate Model
model = aml.ConcreteModel()

In [3]:
#Define Sets
model.directions = aml.Set(initialize=['NS', 'EW'])

#Define Parameters   파라미터 (교통량 데이터 - Austin Traffic Sensor 데이터 가정)
# q: vehicles/hour, s: saturation flow rate (vehicles/hour/lane)
q = {'NS': 800, 'EW': 600}
s = {'NS': 1800, 'EW': 1800}
cycle_time = 120  # Total cycle period (seconds)
loss_time = 10    # Red light and clearance time

In [4]:
# Define Decision Variables   의사결정변수 (녹색 신호 시간)
model.g = aml.Var(model.directions, domain=aml.NonNegativeReals)

In [6]:
# Define Constraints   제약식 (총 사이클 시간과 손실 시간 고려)
def cycle_time_constraint(model):
    return sum(model.g[d] for d in model.directions) + loss_time <= cycle_time
model.cycle_time_con = aml.Constraint(rule=cycle_time_constraint)

def objective_rule(model):
    return sum(q[d] / (s[d] * model.g[d] / cycle_time) for d in model.directions)
model.obj = aml.Objective(rule=objective_rule, sense=aml.minimize)

# Solve the model
solver = aml.SolverFactory('ipopt')
results = solver.solve(model)

# Display results
print("Optimal Green Time (seconds):")
for d in model.directions:
    print(f"{d}: {model.g[d].value:.2f} seconds")
print(f"Total Delay: {model.obj():.2f} vehicle-hours")


(type=<class 'pyomo.core.base.constraint.ScalarConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.AbstractScalarConstraint'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
'pyomo.core.base.objective.ScalarObjective'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.objective.AbstractScalarObjective'>).
This is usually indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
Optimal Green Time (seconds):
NS: 58.95 seconds
EW: 51.05 seconds
Total Delay: 1.69 vehicle-hours
